# Sprint 5 Stage 1 — GPU geometry extraction

Run this notebook from the repository root with the materialized dataset and trained checkpoint present. Set SRC_DIR, V1_DATA_ROOT, V1_CHECKPOINT, and V1_GEOMETRY_OUTPUT in the first cell; the exact configured paths are used. It publishes only the portable geometry cache; analysis happens in the CPU notebook.


In [1]:
from pathlib import Path
import json
import os
import sys

# User parameters: edit these values or provide the corresponding environment variables.
# Server defaults assume the notebook runs from the repository root.
# ---- User paths (edit these one-line values; used exactly) ----
DATASET_ROOT = Path(os.environ.get('V1_DATA_ROOT', '/home/trietlm/anomaly-representation-learning/data/generated/production'))
CHECKPOINT_PATH = Path(os.environ.get('V1_CHECKPOINT', os.environ.get('V1_CHECKPOINT_PATH', '/home/trietlm/anomaly-representation-learning/checkpoints/v1_representation_20260904_01.pt')))
OUTPUT_DIR = Path(os.environ.get('V1_GEOMETRY_OUTPUT', '/home/trietlm/anomaly-representation-learning/experiments/20260905/server-geometry-run-1/geometry-cache'))
if not (DATASET_ROOT / 'manifest.json').is_file():
    raise FileNotFoundError(f"V1 dataset manifest not found at {DATASET_ROOT / 'manifest.json'}. Set V1_DATA_ROOT to the materialized dataset root.")
if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(f"V1 checkpoint not found at {CHECKPOINT_PATH}. Set V1_CHECKPOINT to the trained checkpoint file.")
if not (DATASET_ROOT / 'manifest.json').is_file():
    raise FileNotFoundError(f"V1 dataset manifest not found at {DATASET_ROOT / 'manifest.json'}. Set V1_DATA_ROOT to the materialized dataset root.")
if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(f"V1 checkpoint not found at {CHECKPOINT_PATH}. Set V1_CHECKPOINT to the trained checkpoint file.")
required_splits = ('train', 'val', 'test')
SELECTED_SPLITS = ('test',)
REFERENCE_SPLIT = 'train'
MAX_SAMPLES = 5000
MAX_REFERENCE_SAMPLES = 5000
BATCH_SIZE = 32
SEED = 20260904
SAMPLING = 'head'  # 'head' or deterministic 'reservoir'
DEVICE = 'cuda'
assert set(SELECTED_SPLITS).issubset(set(required_splits))


## Inputs and compatibility
The source package validates the V1 checkpoint schema, fitted normal reference bank, patch geometry, dataset feature width, split names, and all configured bounds before extraction.

In [2]:

# ---- Server paths: repository source (edit these one-line values; used exactly) ----
# Run notebooks from the repository root, or set V1_REPO_ROOT to the checkout path.
V1_REPO_ROOT = os.environ.get('V1_REPO_ROOT', '/home/trietlm/anomaly-representation-learning')
SRC_DIR = os.environ.get('V1_SRC_DIR', str(Path(V1_REPO_ROOT) / 'src'))

_source_dir = Path(SRC_DIR).expanduser()
if not (_source_dir / 'representation').is_dir():
    raise FileNotFoundError(
        f"Repository source not found: SRC_DIR={_source_dir} has no 'representation' package. "
        f"Run from the repository root or set V1_REPO_ROOT / SRC_DIR to the checkout.")
_source_resolved = str(_source_dir.resolve())
if _source_resolved not in sys.path:
    sys.path.insert(0, _source_resolved)
print(f"[Env] Loaded representation modules from: {_source_dir}")

from representation.geometry import DiagnosticConfig, extract_embeddings

config = DiagnosticConfig(
    dataset_root=DATASET_ROOT, checkpoint_path=CHECKPOINT_PATH, output_dir=OUTPUT_DIR,
    splits=SELECTED_SPLITS, reference_split=REFERENCE_SPLIT,
    max_samples=MAX_SAMPLES, max_reference_samples=MAX_REFERENCE_SAMPLES,
    batch_size=BATCH_SIZE, seed=SEED, sampling=SAMPLING, device=DEVICE,
)
print(json.dumps(config.model_dump(mode='json'), indent=2, sort_keys=True))


[Env] Loaded representation modules from: /home/trietlm/anomaly-representation-learning/src


{
  "batch_size": 32,
  "checkpoint_path": "/home/trietlm/anomaly-representation-learning/checkpoints/v1_representation_20260904_01.pt",
  "dataset_root": "/home/trietlm/anomaly-representation-learning/data/generated/production",
  "device": "cuda",
  "max_reference_samples": 5000,
  "max_samples": 5000,
  "output_dir": "/home/trietlm/anomaly-representation-learning/experiments/20260905/server-geometry-run-1/geometry-cache",
  "reference_split": "train",
  "sampling": "head",
  "schema_version": 1,
  "seed": 20260904,
  "splits": [
    "test"
  ]
}


## User-run extraction
The next cell is intentionally unexecuted in the repository. It is the only cell that loads the model/checkpoint and streams selected splits. Keep one batch in memory and do not increase limits beyond available GPU RAM.

In [3]:
result = extract_embeddings(config)
paths = result['paths']
print(f'Published {result["manifest"].records_count} records to {paths.root}')
print('Stage 1 artifacts:', sorted(path.name for path in paths.root.iterdir()))


Published 5000 records to /home/trietlm/anomaly-representation-learning/experiments/20260905/server-geometry-run-1/geometry-cache
Stage 1 artifacts: ['embeddings.npz', 'figures', 'geometry-manifest.json', 'metrics.json', 'neighbors.csv', 'records.csv']


In [4]:
# Reproducibility and handoff check; analysis uses this cache only.
manifest = json.loads(paths.manifest.read_text(encoding='utf-8'))
assert manifest['schema_version'] == 1
assert manifest['records_count'] > 0
for name in ('geometry-manifest.json', 'embeddings.npz', 'records.csv', 'metrics.json', 'neighbors.csv'):
    assert (paths.root / name).is_file(), name
print('Cache ready for Stage 2:', paths.root)


Cache ready for Stage 2: /home/trietlm/anomaly-representation-learning/experiments/20260905/server-geometry-run-1/geometry-cache
